# KWISMO — Modèle A : Scoring de Réputation Temporel & Comportemental

Ce notebook présente l'entraînement et l'évaluation du **Modèle A (LightGBM)** :
1. **Génération du Dataset d'Entraînement Comportemental** (`src.data.generate_model_a_data`).
2. **Entraînement du Modèle LightGBM** (`src.models.model_a.train`).
3. **Visualisations de Performance** (Importances des caractéristiques, Courbe ROC-AUC, Matrice de Confusion).
4. **Validation des Règles de Sécurité Absolues** (Plafonnement du score <= 0.69 en cas de signalement unique).

In [ ]:
# Détection de l'environnement d'exécution
import os
import sys
import subprocess
from pathlib import Path

try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

ON_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists('/kaggle/working')

if ON_COLAB:
    ENV_NAME = "Google Colab"
    if not Path("/content/drive/MyDrive").exists():
        try:
            from google.colab import drive
            print("Connexion automatique à Google Drive...")
            drive.mount('/content/drive')
        except Exception as err:
            print(f"Montage Google Drive recommandé : {err}")
elif ON_KAGGLE:
    ENV_NAME = "Kaggle Notebooks"
else:
    ENV_NAME = "Local"

print(f"Environnement de calcul détecté : {ENV_NAME}")

In [ ]:
# Configuration du dossier du projet et de l'environnement virtuel .venv
current = Path.cwd()
PROJECT_DIR = current
for candidate in [current, current.parent, current.parent.parent]:
    if (candidate / "src").exists() and (candidate / "data").exists():
        PROJECT_DIR = candidate
        break
    elif (candidate / "kwismo-ai" / "src").exists():
        PROJECT_DIR = candidate / "kwismo-ai"
        break

PROJECT_DIR = PROJECT_DIR.resolve()
os.chdir(PROJECT_DIR)

local_venv_win = PROJECT_DIR / ".venv" / "Scripts" / "python.exe"
if local_venv_win.exists():
    PYTHON_BIN = str(local_venv_win)
    site_pkgs = PROJECT_DIR / ".venv" / "Lib" / "site-packages"
    if site_pkgs.exists() and str(site_pkgs) not in sys.path:
        sys.path.insert(0, str(site_pkgs))
else:
    PYTHON_BIN = sys.executable

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

def run_module(module: str) -> None:
    env = os.environ.copy()
    env["PYTHONPATH"] = str(PROJECT_DIR) + os.pathsep + env.get("PYTHONPATH", "")
    result = subprocess.run([PYTHON_BIN, "-m", module], cwd=PROJECT_DIR, capture_output=True, text=True, env=env)
    if result.returncode != 0:
        print(f"Standard Output:\n{result.stdout}")
        print(f"Standard Error:\n{result.stderr}")
        raise RuntimeError(f"{module} a échoué (code {result.returncode})")
    else:
        print(result.stdout)

print("Dossier racine du projet kwismo-ai :", PROJECT_DIR)
print("Interprète Python configuré :", PYTHON_BIN)

## 1. Génération & Entraînement du Modèle A (LightGBM)

Exécution des modules de génération du dataset comportemental et d'entraînement du modèle LightGBM.

In [ ]:
run_module("src.data.generate_model_a_data")
run_module("src.models.model_a.train")

## 2. Métriques Visuelles de Performance (4 Graphiques Parlants)

Génération des graphiques d'analyse des variables déterminantes, de la courbe ROC-AUC et de la matrice de confusion.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay
from src.models.model_a.train import train_model_a
from src.data.features import FEATURE_COLUMNS
from src.models.model_a.predict import predict

sns.set_theme(style="whitegrid", palette="crest")

df_data = pd.read_csv(PROJECT_DIR / "data" / "processed" / "model_a_dataset.csv")
model = train_model_a(df_data)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Feature Importances LightGBM
importances = pd.Series(model.feature_importances_, index=FEATURE_COLUMNS).sort_values(ascending=True)
sns.barplot(x=importances.values, y=importances.index, ax=axes[0, 0], palette="mako")
axes[0, 0].set_title("1. Feature Importances (Importance des Caractéristiques)", fontsize=11, fontweight="bold")
axes[0, 0].set_xlabel("Score d'importance LightGBM")

# 2. Distribution des Vérifications vs Signalements
sns.scatterplot(data=df_data, x="nombre_verifications", y="vitesse_signalements", hue="label", style="label", palette=["#2A9D8F", "#E63946"], ax=axes[0, 1])
axes[0, 1].set_title("2. Vérifications vs Vélocité des Signalements", fontsize=11, fontweight="bold")
axes[0, 1].set_xlabel("Nombre de vérifications")
axes[0, 1].set_ylabel("Signalements / jour")

# 3. Courbe ROC-AUC
y_test_prob = model.predict_proba(df_data[FEATURE_COLUMNS])[:, 1]
fpr, tpr, _ = roc_curve(df_data["label"], y_test_prob)
roc_auc_val = auc(fpr, tpr)
axes[1, 0].plot(fpr, tpr, color="#3A86FF", lw=2, label=f"ROC curve (AUC = {roc_auc_val:.3f})")
axes[1, 0].plot([0, 1], [0, 1], color="grey", linestyle="--")
axes[1, 0].set_title("3. Courbe ROC-AUC (Pouvoir Séparateur du Modèle)", fontsize=11, fontweight="bold")
axes[1, 0].set_xlabel("Taux de Faux Positifs")
axes[1, 0].set_ylabel("Taux de Vrais Positifs")
axes[1, 0].legend(loc="lower right")

# 4. Matrice de Confusion
cm = confusion_matrix(df_data["label"], model.predict(df_data[FEATURE_COLUMNS]))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Sécurisé (0)", "Frauduleux (1)"])
disp.plot(cmap="YlGnBu", ax=axes[1, 1])
axes[1, 1].set_title("4. Matrice de Confusion — Modèle A", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.show()

## 3. Test & Validation des Règles de Sécurité Absolues (Plafonnement <= 0.69 sur Signalement Unique)

Démonstration que quel que soit le contexte, un numéro avec 1 seul signalement ne peut pas dépasser un score de 0.69.

In [ ]:
test_cases = [
    {
        "numero": "237690000001",
        "nombre_signalements": 1,  # 1 seul signalement !
        "nombre_verifications": 50,  # 50 vérifications !
        "categories": {"rep_1": "fake_agent_otp"},
        "description_test": "1 seul signalement avec forte gravité et 50 vérifications"
    },
    {
        "numero": "237690000002",
        "nombre_signalements": 4,  # 4 signalements !
        "nombre_verifications": 20,
        "categories": {"rep_1": "fake_agent_otp", "rep_2": "sim_swap_scam"},
        "description_test": "4 signalements avec catégories graves"
    },
    {
        "numero": "237690000003",
        "nombre_signalements": 0,  # Aucun signalement
        "nombre_verifications": 2,
        "categories": {},
        "description_test": "Numéro ordinaire avec 0 signalement"
    }
]

print("=== Validation des Règles de Sécurité Absolues sur l'Inférence Modèle A ===\n")
for tc in test_cases:
    score, explications, modele_used = predict(tc)
    print(f"Description : {tc['description_test']}")
    print(f"  - Score de Risque calculé : {score} / 1.0 (Modèle : {modele_used})")
    print(f"  - Explications : {explications}")
    print("-" * 70)